# 06 — Model Evaluation: Dual-Head Siamese U-Net

## Objective

This notebook performs evaluation and diagnostic analysis of the trained
**Dual-Head Siamese U-Net** on the untouched test split (933 satellite tiles).

### Key Performance Metrics:
1. **Building Footprint Localization:** IoU, F1, Precision, Recall (pre-disaster building detection).
2. **Damage Classification:** Per-class F1, IoU, Precision, and Recall for `no-damage`, `minor-damage`, `major-damage`, and `destroyed`.
3. **Foreground Macro F1 & mIoU:** Performance strictly across the 4 damage states.
4. **Official xView2 Challenge Score:**
   $$\text{xView2 Score} = 0.3 \times F1_{\text{loc}} + 0.7 \times F1_{\text{dmg\_macro}}$$
5. **Pixel-Level Confusion Matrix:** Normalized error flow across classes.
6. **Qualitative Visual Inspection:** Side-by-side comparison of pre-disaster, post-disaster, ground truth, and predicted masks.



In [ ]:
# ============================================================
# 1. IMPORTS AND ENVIRONMENT SETUP
# ============================================================

import sys
from pathlib import Path
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DEVICE,
    USE_AMP,
    CLASS_MAPPING_5,
    NUM_TOTAL_CLASSES,
    CHECKPOINT_ROOT,
    OUTPUT_ROOT,
)
from src.dataset import get_dataloaders
from src.model import DualHeadSiameseUNet
from src.metrics import calculate_metrics_from_confusion, compute_confusion_matrix
from src.evaluate import plot_confusion_matrix, plot_visual_samples

print(f"Device: {DEVICE}")



In [ ]:
# ============================================================
# 2. LOAD TRAINED MODEL CHECKPOINT
# ============================================================

checkpoint_path = CHECKPOINT_ROOT / "dual_head_siamese_best.pth"
if not checkpoint_path.exists():
    print(f"Best checkpoint not found at {checkpoint_path}. Trying latest checkpoint...")
    checkpoint_path = CHECKPOINT_ROOT / "dual_head_siamese_latest.pth"

assert checkpoint_path.exists(), f"No checkpoint found at {checkpoint_path}!"

print(f"Loading checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

model = DualHeadSiameseUNet(pretrained=False).to(DEVICE)
state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

print("Model successfully loaded in evaluation mode.")
if "best_score" in checkpoint:
    print(f"Best Holdout Score: {checkpoint['best_score']:.4f}")



In [ ]:
# ============================================================
# 3. TEST DATASET INITIALIZATION
# ============================================================

_, _, test_loader = get_dataloaders()
print(f"Test tiles to evaluate: {len(test_loader.dataset)} (Full 1024x1024 tiles)")



In [ ]:
# ============================================================
# 4. RUN TEST EVALUATION
# ============================================================

confusion = np.zeros((NUM_TOTAL_CLASSES, NUM_TOTAL_CLASSES), dtype=np.int64)
visual_samples = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Test Set"):
        pre_img = batch["pre_image"].to(DEVICE, non_blocking=True)
        post_img = batch["post_image"].to(DEVICE, non_blocking=True)
        raw_tgt = batch["raw_target"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP, dtype=torch.float16):
            final_pred, loc_probs, damage_probs = model.predict(pre_img, post_img)

        confusion += compute_confusion_matrix(final_pred, raw_tgt)

        # Collect representative samples for visualization
        if len(visual_samples) < 5 and torch.any(raw_tgt >= 2):
            visual_samples.append({
                "tile_id": batch["tile_id"][0],
                "pre_img": pre_img[0],
                "post_img": post_img[0],
                "gt_mask": raw_tgt[0],
                "pred_mask": final_pred[0],
            })

metrics = calculate_metrics_from_confusion(confusion)
print("\nTest set evaluation completed.")



In [ ]:
# ============================================================
# 5. CLASS-WISE PERFORMANCE TABLE
# ============================================================

print("=" * 75)
print("FINAL TEST SET CLASS-WISE PERFORMANCE")
print("=" * 75)
display(metrics["class_table"])

print("\n" + "=" * 75)
print("CORE METRIC SUMMARY")
print("=" * 75)
print(f"Building Localization F1 : {metrics['loc_f1']:.4f}  (IoU: {metrics['loc_iou']:.4f})")
print(f"Building Localization Prec: {metrics['loc_precision']:.4f}  (Recall: {metrics['loc_recall']:.4f})")
print(f"Damage Macro F1          : {metrics['fg_macro_f1']:.4f}")
print(f"Damage Foreground mIoU   : {metrics['fg_miou']:.4f}")
print(f"Official xView2 Score    : {metrics['xview2_score']:.4f}  (0.3 * Loc_F1 + 0.7 * Dmg_F1)")
print("=" * 75)



In [ ]:
# ============================================================
# 6. CONFUSION MATRIX VISUALIZATION
# ============================================================

cm_path = OUTPUT_ROOT / "test_confusion_matrix.png"
plot_confusion_matrix(confusion, cm_path)

from PIL import Image
plt.figure(figsize=(18, 7))
plt.imshow(Image.open(cm_path))
plt.axis("off")
plt.title("Test Set Confusion Matrix", fontsize=14)
plt.show()



In [ ]:
# ============================================================
# 7. QUALITATIVE VISUAL PREDICTIONS
# ============================================================

if visual_samples:
    vis_path = OUTPUT_ROOT / "test_predictions_visual.png"
    plot_visual_samples(visual_samples, vis_path)
    
    from PIL import Image
    plt.figure(figsize=(18, 4.5 * len(visual_samples)))
    plt.imshow(Image.open(vis_path))
    plt.axis("off")
    plt.title("Sample Test Predictions (Pre, Post, Ground Truth, Predicted)", fontsize=14)
    plt.show()

